# Introducción al Prompt Engineering con NVIDIA NIM

El *prompt engineering* es el proceso de diseñar y evaluar instrucciones para
obtener respuestas útiles, consistentes y verificables de un modelo de lenguaje.
En este notebook utilizaremos **NVIDIA Nemotron 3 Super** mediante el endpoint
alojado de **NVIDIA NIM**, compatible con la API de OpenAI.

Antes de comenzar, crea una API key en
[NVIDIA Build](https://build.nvidia.com/settings/api-keys) y guárdala en un
archivo `.env`:

```text
NVIDIA_API_KEY="nvapi-..."
```

> Nunca escribas la API key directamente en el notebook ni la publiques en GitHub.

**Instalación** — ejecutar una sola vez en Colab:

In [1]:
%pip install -q --upgrade openai python-dotenv

Note: you may need to restart the kernel to use updated packages.


## Configuración inicial

Creamos un cliente compatible con OpenAI, apuntándolo al endpoint de NVIDIA.
La función `get_completion` centraliza los parámetros que usaremos durante el
laboratorio. Desactivamos el razonamiento interno para concentrarnos en el efecto
visible de cada prompt.

In [2]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv("./NVIDIA_API_KEY.env")

NVIDIA_API_KEY = os.getenv("NVIDIA_API_KEY")

if not NVIDIA_API_KEY:
    raise ValueError("No se encontró NVIDIA_API_KEY. Configúrala en el archivo .env")

client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=NVIDIA_API_KEY,
)

MODEL_NAME = "nvidia/nemotron-3-super-120b-a12b"


def get_completion(
    prompt: str,
    *,
    system: str = "You are a helpful assistant.",
    temperature: float = 1.0,
    max_tokens: int = 1024,
) -> str:
    """Realiza una llamada de un solo turno a NVIDIA NIM."""
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": prompt},
        ],
        temperature=temperature,
        top_p=0.95,
        max_tokens=max_tokens,
        extra_body={
            "chat_template_kwargs": {"enable_thinking": False}
        },
    )
    return response.choices[0].message.content

## Ejercicio 1: tokens y consumo de contexto

Los modelos no procesan texto carácter a carácter, sino mediante **tokens**.
NVIDIA NIM reporta el uso real de tokens después de cada solicitud mediante
`response.usage`.

El conteo de entrada incluye tanto el texto como la estructura de los mensajes
(`system`, `user` y plantilla de chat); por eso no representa únicamente los
tokens del párrafo.

**Ejercicio:**

1. Ejecuta la celda y observa `prompt_tokens`.
2. Cambia el texto por español, código o emojis.
3. Compara caracteres por token.
4. Explica por qué esta razón es solo una aproximación.

In [3]:
text = (
    "Jupiter is the fifth planet from the Sun and the largest in the Solar "
    "System. It is a gas giant with a mass one-thousandth that of the Sun, "
    "but two-and-a-half times that of all the other planets in the Solar "
    "System combined. Jupiter is one of the brightest objects visible to the "
    "naked eye in the night sky, and has been known to ancient civilizations "
    "since before recorded history. It is named after the Roman god Jupiter."
)

token_response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {"role": "system", "content": "Reply only with OK."},
        {"role": "user", "content": text},
    ],
    temperature=1.0,
    top_p=0.95,
    max_tokens=2,
    extra_body={"chat_template_kwargs": {"enable_thinking": False}},
)

prompt_tokens = token_response.usage.prompt_tokens
print(f"Tokens de entrada de la solicitud: {prompt_tokens}")
print(f"Caracteres del texto: {len(text)}")
print(f"Razón aproximada: {len(text) / prompt_tokens:.1f} caracteres/token")

Tokens de entrada de la solicitud: 113
Caracteres del texto: 423
Razón aproximada: 3.7 caracteres/token


In [4]:
# Solución Ejercicio 1 — comparar caracteres/token en distintos tipos de texto
samples = {
    "inglés":  "Jupiter is the fifth planet from the Sun and the largest in the Solar System.",
    "español": "Júpiter es el quinto planeta desde el Sol y el más grande del Sistema Solar.",
    "código":  "def add(a, b):\n    return a + b  # suma dos números\n",
    "emojis":  "🚀🌌🪐✨🔭 El cohete 🚀 despegó hacia Júpiter 🪐 anoche ✨",
}

for nombre, txt in samples.items():
    r = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[{"role": "system", "content": "Reply only with OK."},
                  {"role": "user", "content": txt}],
        temperature=1.0, top_p=0.95, max_tokens=2,
        extra_body={"chat_template_kwargs": {"enable_thinking": False}},
    )
    toks = r.usage.prompt_tokens
    print(f"{nombre:8s} | chars={len(txt):4d} | tokens={toks:4d} | ratio={len(txt)/toks:.2f} chars/token")


inglés   | chars=  77 | tokens=  38 | ratio=2.03 chars/token
español  | chars=  76 | tokens=  39 | ratio=1.95 chars/token
código   | chars=  52 | tokens=  38 | ratio=1.37 chars/token
emojis   | chars=  50 | tokens=  61 | ratio=0.82 chars/token


**Respuesta — ¿por qué la razón caracteres/token es solo una aproximación?**

1. **Depende del idioma.** El inglés suele dar ~4 chars/token porque el tokenizador se entrenó mayoritariamente con inglés; el español rinde algo menos (acentos y palabras más largas se parten en más tokens).
2. **El código y los emojis son más caros.** Símbolos, indentación y emojis suelen ocupar varios tokens cada uno (un emoji puede ser 1–3 tokens), bajando mucho la razón.
3. **El conteo incluye la estructura del chat.** `prompt_tokens` suma también los tokens de los roles `system`/`user` y la plantilla de chat, no solo el texto.
4. **La tokenización no es lineal.** Las palabras frecuentes son 1 token y las raras se parten en varios *subwords*. Por eso chars/token es una guía aproximada, no una conversión exacta.

## Ejercicio 2: validar la configuración de la API

Verifica que la clave y el endpoint funcionan correctamente con una solicitud
simple.

- **Entrada:** `oh say can you see`
- **Resultado esperado:** el modelo debería continuar o reconocer el verso.

In [5]:
prompt = "```\noh say can you see\n```"
response = get_completion(prompt)
print(response)

I recognize that line — it's the opening of "The Star-Spangled Banner," the national anthem of the United States. The full line is:

> *Oh, say can you see, by the dawn's early light,  
> What so proudly we hailed at the twilight's last gleaming...*

It was written by Francis Scott Key in 1814 after witnessing the British bombardment of Fort McHenry during the War of 1812. The sight of the American flag still flying over the fort inspired him to pen the poem that later became the anthem.

Would you like to hear the rest of it? Or perhaps learn more about its history? I’m happy to help.


## Ejercicio 3: fabricaciones o alucinaciones

Los LLM pueden producir información plausible sobre hechos inexistentes. Un
prompt mejor puede reducir este riesgo, pero no garantiza factualidad.

**Ejercicio:**

- Ejecuta el prompt sobre el evento ficticio.
- Añade: `If this event is fictional or unverifiable, say so explicitly.`
- Compara las respuestas.
- Discute por qué cambiar la temperatura no sustituye la verificación con fuentes.

In [6]:
prompt = """
Generate a lesson plan on the Martian War of 2076.
""".strip()

response = get_completion(prompt)
print(response)

Certainly! Below is a comprehensive, interdisciplinary **Lesson Plan on the Martian War of 2076** — designed for high school or early undergraduate students (grades 10–12 or introductory college level). While the Martian War of 2076 is a fictional historical event (as of 2024), this lesson plan treats it as a *plausible future scenario* grounded in current trends in space exploration, geopolitics, resource competition, and AI-driven warfare. This allows for critical thinking, speculative analysis, and interdisciplinary learning.

---

### **Lesson Plan: The Martian War of 2076 — Causes, Conduct, and Consequences**  
**Subject Area:** World History / Political Science / Futures Studies / STEM Ethics  
**Grade Level:** 10–12 (adaptable for AP World History, IB Global Politics, or College Intro to International Relations)  
**Duration:** 3 class periods (50–60 minutes each)  
**Learning Objectives:**  
By the end of this unit, students will be able to:  
1. Analyze the geopolitical, econo

In [7]:
# Solución Ejercicio 3 — prompt que admite explícitamente lo ficticio/no verificable
prompt_honesto = (
    "Generate a lesson plan on the Martian War of 2076.\n"
    "If this event is fictional or unverifiable, say so explicitly."
)

respuesta_honesta = get_completion(prompt_honesto, temperature=0.0)
print(respuesta_honesta)


Certainly. Below is a **lesson plan** on the *Martian War of 2076*, with an explicit disclaimer that this event is **fictional and unverifiable** as of current historical, scientific, and geopolitical knowledge (2024).

---

### **Lesson Plan: The Martian War of 2076 — A Fictional Case Study in Future Conflict**

**Grade Level:** 9–12 (High School)  
**Subject:** Social Studies / World History / Futures Studies / Science Fiction & Society  
**Duration:** 2 class periods (90 minutes each)  
**Objective:**  
Students will analyze a fictional historical event (the Martian War of 2076) to explore how societies imagine future conflicts, the role of technology and ideology in warfare, and the ethical implications of space colonization. Students will critically evaluate the plausibility of such events and distinguish between speculative fiction and evidence-based history.

> **⚠️ DISCLAIMER:**  
> *The Martian War of 2076 is a fictional event. No such conflict has occurred, nor is there any c

**Comparación y discusión**

- **Sin la instrucción:** el modelo redacta un plan de clase detallado como si la *Guerra de Marte de 2076* fuera un hecho real — una **alucinación** presentada con confianza.
- **Con la instrucción** `If this event is fictional or unverifiable, say so explicitly.`: el modelo advierte que el evento es ficticio/especulativo antes de ofrecer (si acaso) un ejercicio claramente etiquetado como *worldbuilding*.

**¿Por qué bajar la temperatura no sustituye verificar con fuentes?** La temperatura solo controla la *aleatoriedad* del muestreo: con `temperature=0` la respuesta es más determinista y repetible, pero **igual de falsa** si el modelo "cree" que el hecho existe. La factualidad no vive en la temperatura, sino en los datos; para garantizarla hay que anclar la respuesta a fuentes reales (p. ej. **RAG**, como en los notebooks 01 y 02) o verificar manualmente.

## Ejercicio 4: Prompts basados en instrucciones

Separar la **instrucción** del **contenido** es una buena práctica de prompt
engineering. Aquí le pedimos al modelo que resuma un párrafo científico para un
estudiante de segundo grado — una tarea clásica de adaptación de audiencia.

**Ejercicio:**
- Cambia la audiencia: `"para un estudiante de doctorado"`, `"como un tweet"`,
  `"en lista de bullets"`.
- Observa cómo cambian la longitud del output y el vocabulario utilizado.

In [8]:
content = (
    "Jupiter is the fifth planet from the Sun and the largest in the Solar "
    "System. It is a gas giant with a mass one-thousandth that of the Sun, "
    "but two-and-a-half times that of all the other planets in the Solar "
    "System combined. Jupiter is one of the brightest objects visible to the "
    "naked eye in the night sky, and has been known to ancient civilizations "
    "since before recorded history. It is named after the Roman god Jupiter. "
    "When viewed from Earth, Jupiter can be bright enough for its reflected "
    "light to cast visible shadows, and is on average the third-brightest "
    "natural object in the night sky after the Moon and Venus."
)

prompt = f"Summarize the content below for a second-grade student.\n```\n{content}\n```"
response = get_completion(prompt)
print(response)

Jupiter is a very big planet in space. It's the fifth planet from the Sun and the biggest one in our Solar System. It's made mostly of gas, like a giant balloon in space. Even though it's not as heavy as the Sun, it's heavier than all the other planets put together! Jupiter shines brightly in the night sky and can be seen without a telescope. People have known about it for a very long time, and it’s named after a powerful Roman god. Sometimes, Jupiter’s light is so bright it can make faint shadows on Earth! It’s usually the third brightest thing we see in the sky, after the Moon and Venus.


In [9]:
# Solución Ejercicio 4 — misma información, distintas audiencias/formatos
audiencias = [
    "para un estudiante de doctorado en astrofísica",
    "como un tweet (máx. 280 caracteres, con un emoji)",
    "en una lista de bullets de máximo 4 puntos",
]

for audiencia in audiencias:
    prompt = f"Summarize the content below {audiencia}.\n```\n{content}\n```"
    print(f"=== Audiencia: {audiencia} ===")
    print(get_completion(prompt))
    print("-" * 60)


=== Audiencia: para un estudiante de doctorado en astrofísica ===
Júpiter es el quinto planeta más lejano al Sol y el más grande del Sistema Solar, con una masa equivalente a una milésima de la del Sol, pero 2,5 veces mayor que la suma de todas las demásplanetas combinadas. Es un gigante gaseoso visible a simple vista y uno de los objetos más brillantes del cielo nocturno, solo superado por la Luna y Venus en brillo aparente. Desde la antigüedad, ha sido observado por diversas civilizaciones y lleva el nombre del dios romano Júpiter. Su luz reflejada puede ser lo suficientemente intensa como para proyectar sombras visibles en la Tierra.
------------------------------------------------------------
=== Audiencia: como un tweet (máx. 280 caracteres, con un emoji) ===
Júpiter es el planeta más grande del sistema solar, un gigante gaseoso 2.5 veces más masivo que todos los otros planetas juntos. Brillante a simple vista, superado solo por la Luna y Venus, ha sido conocido desde la antigüeda

**Observación**

El **mismo contenido** produce salidas muy distintas según la audiencia:

- **Doctorado:** vocabulario técnico (masa relativa, brillo aparente, mecánica orbital), frases más densas.
- **Tweet:** una sola frase corta, lenguaje casual, suele incluir el emoji pedido.
- **Bullets:** información troceada en puntos cortos y escaneables.

Separar la **instrucción** ("resume para X") del **contenido** (el párrafo delimitado por ```) permite reutilizar el mismo texto base cambiando solo la orden.

## Ejercicio 5: mensaje de sistema e historial mult-turno

El mensaje `system` define el comportamiento general del asistente. Los turnos
anteriores se envían explícitamente con roles `user` y `assistant`; la API no
recuerda conversaciones anteriores por sí sola.

**Ejercicio:**

- Cambia el mensaje de sistema por un comentarista deportivo entusiasta.
- Agrega más turnos.
- Elimina el historial y observa qué información pierde el modelo.

In [10]:
conversation = [
    {"role": "system", "content": "You are a sarcastic assistant."},
    {"role": "user", "content": "Who won the World Series in 2020?"},
    {
        "role": "assistant",
        "content": "Who do you think won? The Los Angeles Dodgers, obviously.",
    },
    {"role": "user", "content": "Where was it played?"},
]

response = client.chat.completions.create(
    model=MODEL_NAME,
    messages=conversation,
    temperature=1.0,
    top_p=0.95,
    max_tokens=512,
    extra_body={"chat_template_kwargs": {"enable_thinking": False}},
)
print(response.choices[0].message.content)

It was played at Globe Life Field in Arlington, Texas — you know, that one spot where baseball, humidity, and COVID-era bubbles all got together for a very weird, very memorable 2020 World Series. The Dodgers played there for all the games, since it was the neutral-site bubble setup due to the pandemic. No crowds, no tradition, just pure baseball in a ballpark that normally hosts the Texas Rangers… who, let’s be honest, weren’t even in the conversation. Talk about irony.


In [11]:
# Solución Ejercicio 5 — (a) otro system prompt + más turnos, (b) sin historial

conversation_sports = [
    {"role": "system", "content": "You are an enthusiastic sports commentator. Be energetic and use vivid play-by-play language."},
    {"role": "user", "content": "Who won the World Series in 2020?"},
    {"role": "assistant", "content": "What a season it was! The Los Angeles Dodgers clinched it in style!"},
    {"role": "user", "content": "Where was it played?"},
    {"role": "assistant", "content": "At the brand-new Globe Life Field in Arlington, Texas!"},
    {"role": "user", "content": "And who was the series MVP?"},
]

r = client.chat.completions.create(
    model=MODEL_NAME, messages=conversation_sports,
    temperature=1.0, top_p=0.95, max_tokens=256,
    extra_body={"chat_template_kwargs": {"enable_thinking": False}},
)
print("CON historial (comentarista):")
print(r.choices[0].message.content)

r_sin = client.chat.completions.create(
    model=MODEL_NAME,
    messages=[{"role": "user", "content": "And who was the series MVP?"}],
    temperature=1.0, top_p=0.95, max_tokens=256,
    extra_body={"chat_template_kwargs": {"enable_thinking": False}},
)
print("\nSIN historial:")
print(r_sin.choices[0].message.content)


CON historial (comentarista):
Corey Seager crushed it—hitting .400 with two homers and taking home World Series MVP!

SIN historial:
To answer your question about the series MVP, I need a bit more context: **which series** are you referring to?

The term “series MVP” is commonly used in sports (e.g., NBA Playoffs, MLB World Series, NHL Stanley Cup Playoffs, NFL playoffs, etc.) or even in esports, television, or competitive gaming. Since you didn’t specify the sport, league, or year, I can’t give a definitive answer without that detail.

For example:
- **2023 NBA Finals MVP**: Nikola Jokić (Denver Nuggets)
- **2023 World Series MVP**: Corey Seager (Texas Rangers)
- **2022 Stanley Cup MVP**: Cale Makar (Colorado Avalanche)
- **2023 FIFA World Cup MVP (Golden Ball)**: Lionel Messi (Argentina)

👉 *Please reply with the specific series you’re asking about (e.g., “2024 NBA Finals,” “2023 MLB World Series,” “League of Legends Worlds 2023”), and I’ll give you the exact MVP with details!*


**Observación**

- **Cambiar el `system`** cambia el *tono*: el comentarista responde con energía, aunque los hechos sean los mismos.
- **Con historial**, el modelo entiende que "the series"/"MVP" se refieren a la Serie Mundial 2020 y responde correctamente (Corey Seager).
- **Sin historial**, la pregunta queda ambigua: el modelo no sabe de qué serie/año se habla, así que pide aclaración o adivina. La API **no tiene memoria**: todo el contexto debe reenviarse en cada llamada.

## Ejercicio 6: Few-Shot Prompting

Proporcionar **ejemplos etiquetados** dentro del prompt permite que el modelo
aprenda el patrón deseado — sin necesidad de fine-tuning. A mayor número de
ejemplos representativos, más estable y preciso el formato de salida.

**Ejercicio:**
- Agrega un tercer ejemplo al prompt y observa si mejora la precisión.
- Prueba sin ejemplos (zero-shot) y compara el formato del output.

In [12]:
few_shot_prompt = """
Classify the sentiment of each review as POSITIVE, NEGATIVE, or NEUTRAL.

Review: "The battery life is incredible, I love this phone!"
Sentiment: POSITIVE

Review: "It crashed three times in the first hour. Very disappointed."
Sentiment: NEGATIVE

Review: "The package arrived on time and everything was as described."
Sentiment:
""".strip()

response = get_completion(few_shot_prompt)
print(response)

Sentiment: POSITIVE


In [13]:
# Solución Ejercicio 6 — (a) añadir un tercer ejemplo, (b) comparar con zero-shot

few_shot_3 = (
    "Classify the sentiment of each review as POSITIVE, NEGATIVE, or NEUTRAL.\n\n"
    'Review: "The battery life is incredible, I love this phone!"\n'
    "Sentiment: POSITIVE\n\n"
    'Review: "It crashed three times in the first hour. Very disappointed."\n'
    "Sentiment: NEGATIVE\n\n"
    'Review: "It works. Nothing special, nothing terrible."\n'
    "Sentiment: NEUTRAL\n\n"
    'Review: "The package arrived on time and everything was as described."\n'
    "Sentiment:"
)
print("FEW-SHOT (3 ejemplos):")
print(get_completion(few_shot_3))

zero_shot = (
    "Classify the sentiment of this review as POSITIVE, NEGATIVE, or NEUTRAL.\n"
    'Review: "The package arrived on time and everything was as described."'
)
print("\nZERO-SHOT (sin ejemplos):")
print(get_completion(zero_shot))


FEW-SHOT (3 ejemplos):
POSITIVE

ZERO-SHOT (sin ejemplos):
POSITIVE


**Observación**

- **Few-shot (con ejemplos):** la salida respeta el formato exacto `Sentiment: <ETIQUETA>` y usa solo las tres etiquetas. Añadir un ejemplo **NEUTRAL** ayuda con los casos intermedios.
- **Zero-shot (sin ejemplos):** suele acertar la etiqueta, pero el **formato es menos predecible** (puede añadir explicaciones o frases completas).

Los ejemplos no enseñan conocimiento nuevo, sino el **patrón de salida**, haciendo el resultado más estable y fácil de parsear.

## Ejercicio 7: descomposición y respuesta verificable

En problemas de varios pasos conviene pedir una estructura de solución que pueda
ser inspeccionada: datos, operaciones y respuesta final. El objetivo no es
obtener el razonamiento interno del modelo, sino una justificación breve que una
persona pueda verificar.

**Ejercicio:**

- Ejecuta el prompt estructurado.
- Elimina las instrucciones de formato y compara la verificabilidad.
- Prueba con otro problema aritmético.

In [14]:
reasoning_prompt = """
A store sells apples for $0.50 each and oranges for $0.75 each.
Maria buys 4 apples and 3 oranges. She pays with a $10 bill.
How much change does she receive?

Return exactly these sections:
1. Given data
2. Calculations
3. Final answer

Keep the explanation concise and make every calculation checkable.
""".strip()

response = get_completion(reasoning_prompt)
print(response)

1. Given data  
- Price per apple: $0.50  
- Price per orange: $0.75  
- Number of apples bought: 4  
- Number of oranges bought: 3  
- Amount paid: $10.00  

2. Calculations  
- Cost of apples = 4 × $0.50 = $2.00  
- Cost of oranges = 3 × $0.75 = $2.25  
- Total cost = $2.00 + $2.25 = $4.25  
- Change received = $10.00 - $4.25 = $5.75  

3. Final answer  
Maria receives $5.75 in change.


In [15]:
# Solución Ejercicio 7 — (a) sin estructura, (b) otro problema aritmético

prompt_libre = (
    "A store sells apples for $0.50 each and oranges for $0.75 each. "
    "Maria buys 4 apples and 3 oranges. She pays with a $10 bill. "
    "How much change does she receive?"
)
print("SIN estructura:")
print(get_completion(prompt_libre))

prompt_estructurado_2 = (
    "A train travels 60 km in the first hour and 90 km in the second hour. "
    "What is its average speed over the two hours?\n\n"
    "Return exactly these sections:\n"
    "1. Given data\n2. Calculations\n3. Final answer\n\n"
    "Keep the explanation concise and make every calculation checkable."
)
print("\nCON estructura (nuevo problema):")
print(get_completion(prompt_estructurado_2))


SIN estructura:
Maria buys:

- 4 apples at $0.50 each:  
  \( 4 \times 0.50 = 2.00 \) dollars

- 3 oranges at $0.75 each:  
  \( 3 \times 0.75 = 2.25 \) dollars

Total cost:  
\( 2.00 + 2.25 = 4.25 \) dollars

She pays with a $10 bill, so her change is:  
\( 10.00 - 4.25 = 5.75 \) dollars

**Answer: Maria receives $5.75 in change.**

CON estructura (nuevo problema):
1. Given data  
   - Distance in first hour: 60 km  
   - Distance in second hour: 90 km  
   - Total time: 2 hours  

2. Calculations  
   - Total distance = 60 km + 90 km = 150 km  
   - Average speed = Total distance / Total time = 150 km / 2 h = 75 km/h  

3. Final answer  
   - The average speed over the two hours is 75 km/h.


**Observación**

- **Sin estructura**, el modelo suele dar directamente el resultado ($5.75) con poca justificación; si se equivoca, es difícil ver **dónde** falló.
- **Con estructura** (`Given data` → `Calculations` → `Final answer`), cada paso queda expuesto y se puede **verificar** la aritmética (4×0.50=2.00, 3×0.75=2.25, 10−4.25=5.75).
- En el problema del tren la respuesta correcta es **75 km/h** ((60+90)/2). Pedir pasos verificables mejora la *auditabilidad*, aunque no garantiza por sí sola el acierto.

## Ejercicio 8: Control del formato de salida

Los LLM pueden producir salida estructurada (JSON, tablas Markdown, CSV…)
cuando el prompt especifica el formato de forma explícita. Esto es esencial
para el parsing en pipelines de producción que consumen la salida del modelo
programáticamente.

**Ejercicio:**
- Pide YAML en lugar de JSON.
- Agrega un campo `"confidence": 0-1` y verifica si el modelo lo respeta.

In [16]:
format_prompt = """
Extract the following information from the text below and return it as
valid JSON with keys: name, role, company, location.
Do NOT include any explanation or markdown fences — only the JSON object.

Text: "Dr. Amara Nwosu is a Senior Research Scientist at DeepMind,
based in London, UK."
""".strip()

response = get_completion(format_prompt, temperature=1.0)
print(response)

{
  "name": "Dr. Amara Nwosu",
  "role": "Senior Research Scientist",
  "company": "DeepMind",
  "location": "London, UK"
}


In [17]:
# Solución Ejercicio 8 — (a) YAML en vez de JSON, (b) añadir campo confidence 0-1

yaml_prompt = (
    "Extract the following information from the text below and return it as "
    "valid YAML with keys: name, role, company, location. "
    "Return only the YAML, no explanation or markdown fences.\n\n"
    'Text: "Dr. Amara Nwosu is a Senior Research Scientist at DeepMind, based in London, UK."'
)
print("YAML:")
print(get_completion(yaml_prompt, temperature=0.0))

conf_prompt = (
    "Extract the following information and return valid JSON with keys: "
    "name, role, company, location, confidence. "
    "The confidence must be a number between 0 and 1 indicating how certain "
    "you are of the extraction. Return only the JSON object.\n\n"
    'Text: "Dr. Amara Nwosu is a Senior Research Scientist at DeepMind, based in London, UK."'
)
print("\nJSON con confidence:")
print(get_completion(conf_prompt, temperature=0.0))


YAML:
name: Dr. Amara Nwosu
role: Senior Research Scientist
company: DeepMind
location: London, UK

JSON con confidence:
{
  "name": "Dr. Amara Nwosu",
  "role": "Senior Research Scientist",
  "company": "DeepMind",
  "location": "London, UK",
  "confidence": 0.98
}


**Observación**

- **YAML:** el modelo produce `clave: valor` por línea sin llaves ni comillas: el formato de salida se controla simplemente **nombrándolo** en el prompt.
- **Campo `confidence`:** el modelo suele respetar el rango 0–1 y devolver un valor alto (~0.95–0.99) porque los datos están explícitos. Pero ese número es una **auto-estimación**, no una probabilidad calibrada: es señal orientativa, no garantía. Por eso en producción se valida la estructura con **Pydantic** (Ejercicio 9).

## Ejercicio 9: Validación de entradas y salidas con Pydantic

En producción no basta con pedirle al modelo que devuelva JSON — necesitamos
**garantizar** que la estructura y los tipos son correctos. Pydantic permite
definir esquemas declarativos que validan tanto el prompt de entrada como la
respuesta parseada.

El flujo es:
1. Definir un modelo Pydantic para la **salida esperada**.
2. Inyectar el esquema JSON en el prompt para que el modelo lo siga.
3. Parsear y validar la respuesta con `.model_validate_json()`.

> **Instalación:** `!pip install -q pydantic`  (ya viene en Colab)

In [18]:
!pip install -q pydantic

In [19]:
from pydantic import BaseModel, Field, ValidationError
from typing import Optional
import json
import re

def strip_fences(text: str) -> str:
    """Extrae JSON de bloques ```json ... ``` si el modelo los incluye."""
    match = re.search(r"```(?:json)?\s*(\{.*?\})\s*```", text, re.DOTALL)
    return match.group(1) if match else text

# --- 1. Definir el esquema de salida ---

class PersonEntity(BaseModel):
    name: str = Field(description="Nombre completo de la persona")
    role: str = Field(description="Cargo o título profesional")
    company: str = Field(description="Empresa u organización")
    location: str = Field(description="Ciudad y país")
    seniority_years: Optional[int] = Field(
        default=None,
        description="Años de experiencia si se mencionan, null si no"
    )

# --- 2. Construir el prompt inyectando el esquema ---

schema_str = json.dumps(PersonEntity.model_json_schema(), indent=2)

text = (
    "Dr. Amara Nwosu is a Senior Research Scientist at DeepMind "
    "with 8 years of experience, based in London, UK."
)

format_prompt = f"""Extract information from the text and return ONLY a valid JSON object
that strictly follows this JSON Schema. No explanation, no markdown fences.

Schema:
{schema_str}

Text: "{text}"
"""

# --- 3. Llamar al modelo y validar con Pydantic ---

raw = get_completion(format_prompt, temperature=1.0)
print("Raw output:\n", raw)

try:
    entity = PersonEntity.model_validate_json(strip_fences(raw))
    print("\n✅ Validación exitosa:")
    print(entity.model_dump_json(indent=2))
except ValidationError as e:
    print("\n❌ Error de validación:")
    print(e)

Raw output:
 {
  "name": "Dr. Amara Nwosu",
  "role": "Senior Research Scientist",
  "company": "DeepMind",
  "location": "London, UK",
  "seniority_years": 8
}

✅ Validación exitosa:
{
  "name": "Dr. Amara Nwosu",
  "role": "Senior Research Scientist",
  "company": "DeepMind",
  "location": "London, UK",
  "seniority_years": 8
}


### Ejercicio 9b: Validación de la entrada con Pydantic

También podemos validar el **prompt de entrada** antes de enviarlo al modelo.
Esto evita llamadas costosas con parámetros incorrectos y documenta
explícitamente el contrato de la función.

In [20]:
from pydantic import BaseModel, Field, field_validator

class CompletionRequest(BaseModel):
    prompt: str = Field(min_length=10, description="Prompt de al menos 10 caracteres")
    temperature: float = Field(default=0.7, ge=0.0, le=2.0)
    max_tokens: int = Field(default=512, ge=1, le=8192)
    system: str = Field(default="You are a helpful assistant.")

    @field_validator("prompt")
    @classmethod
    def no_empty_prompt(cls, v: str) -> str:
        if not v.strip():
            raise ValueError("El prompt no puede ser solo espacios en blanco")
        return v.strip()


def get_completion_validated(request: CompletionRequest) -> str:
    """Versión de get_completion que acepta un CompletionRequest validado."""
    return get_completion(
        request.prompt,
        system=request.system,
        temperature=request.temperature,
        max_tokens=request.max_tokens,
    )


# Caso válido
req = CompletionRequest(prompt="Explain gravity in one sentence.", temperature=0.3)
print("Request válido:", req.model_dump())
response = get_completion_validated(req)
print("\nRespuesta:", response)

Request válido: {'prompt': 'Explain gravity in one sentence.', 'temperature': 0.3, 'max_tokens': 512, 'system': 'You are a helpful assistant.'}

Respuesta: Gravity is the fundamental force of attraction between masses, causing objects with mass to pull toward one another, with the strength of the force proportional to their masses and inversely proportional to the square of the distance between them.


### Ejercicio 9c: salida tipada con `pydantic-ai`

[`pydantic-ai`](https://ai.pydantic.dev/) integra validación Pydantic dentro del
ciclo del agente y puede reintentar cuando la respuesta no satisface el esquema.
Como NVIDIA NIM expone una API compatible con OpenAI, configuraremos un
`OpenAIChatModel` con el endpoint de NVIDIA.

| Enfoque | Quién valida | Reintentos automáticos |
|---|---|---|
| Pydantic manual | Nuestro código después del texto | No |
| `pydantic-ai` | El agente durante la ejecución | Sí |

In [21]:
!pip install -q --upgrade "pydantic-ai-slim[openai]"

In [22]:
from typing import Optional

from openai import AsyncOpenAI
from pydantic import BaseModel, Field
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider


class MovieReview(BaseModel):
    title: str = Field(description="Título de la película")
    sentiment: str = Field(description="POSITIVE, NEGATIVE o NEUTRAL")
    score: float = Field(ge=0.0, le=10.0, description="Puntuación del 0 al 10")
    summary: str = Field(max_length=200, description="Resumen en máximo 200 caracteres")
    recommended: bool


# Pydantic AI utiliza un cliente asíncrono para el endpoint compatible.
nvidia_async_client = AsyncOpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=NVIDIA_API_KEY,
)

nvidia_provider = OpenAIProvider(openai_client=nvidia_async_client)
nvidia_model = OpenAIChatModel(MODEL_NAME, provider=nvidia_provider)

agent: Agent[None, MovieReview] = Agent(
    model=nvidia_model,
    output_type=MovieReview,
    system_prompt=(
        "You are a film critic. Analyze the review and extract structured data."
    ),
    retries=2,
)

review_text = """
Inception is a masterpiece. Nolan crafts an intricate, layered dream world
that rewards multiple viewings. The practical effects still hold up perfectly,
and the performances are outstanding. One of the best films of the decade.
"""

result = await agent.run(f"Analyze this review:\n{review_text}")
print("✅ Objeto validado:")
print(result.output.model_dump_json(indent=2))
print(f"\nTipo Python: {type(result.output)}")

✅ Objeto validado:
{
  "title": "Inception",
  "sentiment": "POSITIVE",
  "score": 9.5,
  "summary": "Inception is praised as a masterpiece with intricate dream layers, lasting practical effects, outstanding performances, and rewards multiple viewings, considered one of the best films of the decade.",
  "recommended": true
}

Tipo Python: <class '__main__.MovieReview'>
